# Synthetic EDA for Blast-to-Truck-to-Crusher Traceability

**Course:** Research Methods and Scientific Integrity in AI and Advanced Technologies  
**Program:** Doctoral Program in Deep Tech with a focus on Artificial Intelligence and Emerging Technologies, Universidad Nacional Mayor de San Marcos  
**Project:** Probabilistic, Adaptive, and Interpretable Prediction of Primary Crusher Energy Demand from Blast-Design and Material-Traceability Data

## Scientific integrity statement

This notebook does **not** use raw operational data. It generates synthetic records calibrated from verified aggregate statistics obtained from the operational systems before access was discontinued.

The purpose is to demonstrate a reproducible, auditable EDA workflow. Row-level records, coordinates, equipment identifiers, timestamps, and signal values are synthetic and must not be interpreted as actual operational measurements.

## 1. Verified aggregate inputs

| Year | Unique fired blasts | Blast-hole records | Min holes/blast | Max holes/blast | Mean | Median |
|---:|---:|---:|---:|---:|---:|---:|
| 2024 | 1,070 | 154,821 | 1 | 907 | 144.692523 | 112 |
| 2025 | 1,039 | 180,018 | 4 | 1,297 | 173.260827 | 140 |
| 2026, to 17 June | 516 | 77,704 | 4 | 779 | 150.589147 | 123 |

**Total analytical period:** 1 January 2024 to 17 June 2026  
**Total unique fired blasts:** 2,625  
**Total blast-hole records:** 412,543  
**Weighted mean holes per blast:** 157.16

Traceability design: `blast_name -> polygon/source -> truck cycle/LOADID -> dump timestamp -> selected primary crusher -> crusher-power signal`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

cwd = Path.cwd()
if cwd.name == "notebooks":
    EDA_ROOT = cwd.parent
elif (cwd / "eda").exists():
    EDA_ROOT = cwd / "eda"
else:
    EDA_ROOT = cwd

DATA_DIR = EDA_ROOT / "data" / "synthetic"
FIG_DIR = EDA_ROOT / "figures"
REPORT_DIR = EDA_ROOT / "reports"
for p in [DATA_DIR, FIG_DIR, REPORT_DIR]: p.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
print("EDA root:", EDA_ROOT)

In [ ]:
AGGREGATES = {
    2024: {"blasts": 1070, "holes": 154821, "min": 1, "max": 907, "mean": 144.692523, "median": 112},
    2025: {"blasts": 1039, "holes": 180018, "min": 4, "max": 1297, "mean": 173.260827, "median": 140},
    2026: {"blasts": 516, "holes": 77704, "min": 4, "max": 779, "mean": 150.589147, "median": 123},
}
STUDY_START = "2024-01-01"
STUDY_END = "2026-06-17"

aggregate_summary = pd.DataFrame([
    {"year": y, "unique_blasts": c["blasts"], "blast_holes": c["holes"],
     "min_holes_per_blast": c["min"], "max_holes_per_blast": c["max"],
     "mean_holes_per_blast": c["mean"], "median_holes_per_blast": c["median"]}
    for y, c in AGGREGATES.items()
])
aggregate_summary

## 2. Synthetic data-generation functions

The generator creates three synthetic tables:

1. `synthetic_blasts.csv`
2. `synthetic_truck_cycles.csv`
3. `synthetic_crusher_signals.csv`

The blast table is calibrated to match the verified yearly number of blasts and blast-hole records exactly.

In [ ]:
def allocate_holes_exact(n, total, min_value, max_value, median_hint, rng):
    values = np.clip(np.rint(rng.lognormal(np.log(max(median_hint, 1)), 0.75, n)), min_value, max_value).astype(int)
    values[0], values[1] = min_value, max_value
    diff = int(total - values.sum())
    adjustable = np.arange(2, n)
    attempts = 0
    while diff != 0 and attempts < 250000:
        attempts += 1
        if diff > 0:
            cand = adjustable[values[adjustable] < max_value]
            idx = rng.choice(cand)
            step = min(diff, max_value - values[idx], int(rng.integers(1, 8)))
            values[idx] += step; diff -= step
        else:
            cand = adjustable[values[adjustable] > min_value]
            idx = rng.choice(cand)
            step = min(-diff, values[idx] - min_value, int(rng.integers(1, 8)))
            values[idx] -= step; diff += step
    if values.sum() != total:
        raise RuntimeError(f"Could not match total holes. Got {values.sum()}, expected {total}.")
    return values

def generate_blasts(seed=SEED):
    rng = np.random.default_rng(seed); rows = []
    for year, cfg in AGGREGATES.items():
        holes = allocate_holes_exact(cfg['blasts'], cfg['holes'], cfg['min'], cfg['max'], cfg['median'], rng)
        dates = pd.date_range(f"{year}-01-01", STUDY_END if year == 2026 else f"{year}-12-31", periods=cfg['blasts'])
        rng.shuffle(holes)
        for i, (date, holes_count) in enumerate(zip(dates, holes), start=1):
            rows.append({
                "blast_name": f"BLST_{year}_{i:04d}", "year": year,
                "firing_datetime": date + pd.Timedelta(hours=int(rng.integers(6, 22)), minutes=int(rng.integers(0, 60))),
                "bench": int(rng.choice(np.arange(3600, 4305, 15))),
                "x_centroid_anon": float(rng.normal(0, 850)), "y_centroid_anon": float(rng.normal(0, 850)),
                "z_bench_anon": float(rng.normal(0, 120)), "holes_count": int(holes_count), "blast_status": "BLASTED"
            })
    return pd.DataFrame(rows)

def generate_truck_cycles(blasts, seed=SEED):
    rng = np.random.default_rng(seed + 1); rows = []; load_counter = 1
    for _, b in blasts.iterrows():
        tonnes_est = max(300, b["holes_count"] * rng.uniform(900, 1500))
        n_trucks = int(np.clip(np.ceil(tonnes_est / rng.uniform(330, 380)), 1, 120))
        crusher = rng.choice(["Chancadora 01", "Chancadora 02"], p=[0.56, 0.44])
        origin_type = rng.choice(["POLYGON_DIRECT", "STOCKPILE", "AMBIGUOUS"], p=[0.78, 0.15, 0.07])
        for _ in range(n_trucks):
            load_dt = pd.Timestamp(b["firing_datetime"]) + pd.Timedelta(days=int(rng.integers(0, 8)), hours=float(rng.uniform(0, 24)))
            dump_dt = load_dt + pd.Timedelta(minutes=float(rng.uniform(18, 95)))
            payload = float(np.clip(rng.normal(350, 22), 280, 430))
            has_payload = rng.random() > 0.025; has_time = rng.random() > 0.018
            confidence = "LOW" if origin_type in ["STOCKPILE", "AMBIGUOUS"] else ("MEDIUM" if rng.random() < 0.10 else "HIGH")
            rows.append({
                "LOADID": f"L{load_counter:010d}", "DUMPID": f"D{load_counter:010d}", "blast_name": b["blast_name"],
                "truck_id_anon": f"TRK_{int(rng.integers(1, 80)):03d}", "shovel_id_anon": f"SHV_{int(rng.integers(1, 14)):02d}",
                "selected_crusher": crusher, "load_datetime": load_dt if has_time else pd.NaT, "dump_datetime": dump_dt if has_time else pd.NaT,
                "payload_tonnes": payload if has_payload else np.nan, "origin_type": origin_type, "destination_type": "PRIMARY_CRUSHER",
                "traceability_confidence": confidence, "material_category": rng.choice(["SULFIDE", "OXIDE", "WASTE"], p=[0.72, 0.08, 0.20])
            })
            load_counter += 1
    return pd.DataFrame(rows)

def generate_crusher_signals(trucks, seed=SEED):
    rng = np.random.default_rng(seed + 2)
    valid_times = trucks["dump_datetime"].dropna()
    timestamps = pd.date_range(valid_times.min().floor("H"), valid_times.max().ceil("H"), freq="5min")
    rows = []
    for crusher in ["Chancadora 01", "Chancadora 02"]:
        base = 1650 if crusher == "Chancadora 01" else 1520
        for ts in timestamps:
            power = max(0, base + 80*np.sin((ts.hour/24)*2*np.pi) + rng.normal(0, 110))
            if rng.random() < 0.015: power += rng.uniform(700, 1800)
            state = "RUNNING" if rng.random() > 0.06 else "STOPPED"
            quality = rng.choice(["GOOD", "NO_DATA", "BAD_INPUT", "CALC_FAILED"], p=[0.955, 0.025, 0.012, 0.008])
            if state == "STOPPED": power = rng.uniform(0, 80)
            if quality != "GOOD": power = np.nan
            rows.append({"timestamp": ts, "crusher": crusher, "power_kw": power, "operational_state": state, "signal_quality": quality})
    return pd.DataFrame(rows)

In [ ]:
blasts = generate_blasts()
trucks = generate_truck_cycles(blasts)
signals = generate_crusher_signals(trucks)

blasts.to_csv(DATA_DIR / "synthetic_blasts.csv", index=False)
trucks.to_csv(DATA_DIR / "synthetic_truck_cycles.csv", index=False)
signals.to_csv(DATA_DIR / "synthetic_crusher_signals.csv", index=False)

print("blasts", blasts.shape)
print("trucks", trucks.shape)
print("signals", signals.shape)

## 3. Validation against verified aggregate statistics

This checkpoint confirms that the synthetic blast table preserves the real aggregate structure.

In [ ]:
validation = (blasts.groupby("year")
    .agg(CANTIDAD_VOLADURAS=("blast_name", "nunique"), CANTIDAD_TALADROS=("holes_count", "sum"),
         MIN_TALADROS=("holes_count", "min"), MAX_TALADROS=("holes_count", "max"),
         PROMEDIO_TALADROS=("holes_count", "mean"), MEDIANA_TALADROS=("holes_count", "median"))
    .reset_index())
validation

## 4. Blast inventory EDA

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(validation["year"].astype(str), validation["CANTIDAD_VOLADURAS"])
ax.set_title("Unique fired blasts by year")
ax.set_xlabel("Year"); ax.set_ylabel("Number of blasts")
fig.tight_layout(); fig.savefig(FIG_DIR / "01_blasts_by_year.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(validation["year"].astype(str), validation["CANTIDAD_TALADROS"])
ax.set_title("Blast-hole records by year")
ax.set_xlabel("Year"); ax.set_ylabel("Number of blast-hole records")
fig.tight_layout(); fig.savefig(FIG_DIR / "02_holes_by_year.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(blasts["holes_count"], bins=60)
ax.set_title("Distribution of holes per blast")
ax.set_xlabel("Holes per blast"); ax.set_ylabel("Synthetic blast count")
fig.tight_layout(); fig.savefig(FIG_DIR / "03_holes_per_blast_distribution.png", dpi=150); plt.show()

In [ ]:
small_blasts = blasts[blasts["holes_count"] < 5]
small_blast_summary = small_blasts.groupby("year").agg(small_blasts=("blast_name", "count"), min_holes=("holes_count", "min"), max_holes=("holes_count", "max")).reset_index()
small_blast_summary

## 5. Truck-cycle EDA and traceability audit

In [ ]:
truck_inventory = pd.DataFrame({
    "metric": ["truck_cycles", "unique_blasts_represented", "valid_payload_pct", "valid_dump_timestamp_pct", "mean_payload_tonnes", "median_payload_tonnes"],
    "value": [len(trucks), trucks["blast_name"].nunique(), trucks["payload_tonnes"].notna().mean()*100,
              trucks["dump_datetime"].notna().mean()*100, trucks["payload_tonnes"].mean(), trucks["payload_tonnes"].median()]
})
truck_inventory

In [ ]:
crusher_counts = trucks["selected_crusher"].value_counts().rename_axis("crusher").reset_index(name="truck_cycles")
crusher_counts["share_pct"] = crusher_counts["truck_cycles"] / crusher_counts["truck_cycles"].sum() * 100
crusher_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(crusher_counts["crusher"], crusher_counts["truck_cycles"])
ax.set_title("Synthetic truck cycles by primary crusher")
ax.set_xlabel("Primary crusher"); ax.set_ylabel("Truck cycles")
fig.tight_layout(); fig.savefig(FIG_DIR / "04_truck_cycles_by_crusher.png", dpi=150); plt.show()

In [ ]:
traceability = (trucks.groupby(["origin_type", "traceability_confidence"])
    .agg(truck_cycles=("LOADID", "count"), blasts=("blast_name", "nunique"),
         valid_payload_pct=("payload_tonnes", lambda s: s.notna().mean()*100),
         valid_dump_time_pct=("dump_datetime", lambda s: s.notna().mean()*100))
    .reset_index())
traceability

In [ ]:
confidence_counts = trucks["traceability_confidence"].value_counts().rename_axis("confidence").reset_index(name="truck_cycles")
confidence_counts["share_pct"] = confidence_counts["truck_cycles"] / confidence_counts["truck_cycles"].sum() * 100
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(confidence_counts["confidence"], confidence_counts["share_pct"])
ax.set_title("Traceability confidence distribution")
ax.set_xlabel("Traceability confidence"); ax.set_ylabel("Share of truck cycles (%)")
fig.tight_layout(); fig.savefig(FIG_DIR / "05_traceability_confidence.png", dpi=150); plt.show()
confidence_counts

## 6. Primary crusher scope selection

The initial confirmatory study should use one crusher to reduce heterogeneity and simplify temporal alignment.

In [ ]:
primary_scope = (trucks.groupby("selected_crusher")
    .agg(truck_cycles=("LOADID", "count"), blasts=("blast_name", "nunique"),
         direct_polygon_cycles=("origin_type", lambda s: (s == "POLYGON_DIRECT").sum()),
         high_confidence_cycles=("traceability_confidence", lambda s: (s == "HIGH").sum()),
         valid_payload_pct=("payload_tonnes", lambda s: s.notna().mean()*100),
         valid_dump_time_pct=("dump_datetime", lambda s: s.notna().mean()*100),
         total_payload_tonnes=("payload_tonnes", "sum"))
    .reset_index())
primary_scope["high_confidence_share_pct"] = primary_scope["high_confidence_cycles"] / primary_scope["truck_cycles"] * 100
selected_crusher = primary_scope.sort_values(["high_confidence_cycles", "valid_payload_pct", "valid_dump_time_pct"], ascending=False).iloc[0]["selected_crusher"]
print("Recommended crusher:", selected_crusher)
primary_scope

## 7. Crusher signal EDA

The synthetic signal table uses five-minute observations to keep the repository small. The real historian visualisation was reported as second-level; final extraction must document whether data are raw, compressed, interpolated, calculated, or exception-based.

In [ ]:
signal_quality = signals.groupby(["crusher", "signal_quality"]).agg(records=("timestamp", "count")).reset_index()
signal_quality["share_pct"] = signal_quality.groupby("crusher")["records"].transform(lambda s: s/s.sum()*100)
signal_quality

In [ ]:
power_summary = (signals[signals["signal_quality"] == "GOOD"].groupby("crusher")
    .agg(records=("power_kw", "count"), mean_kw=("power_kw", "mean"), median_kw=("power_kw", "median"),
         p95_kw=("power_kw", lambda s: np.percentile(s.dropna(), 95)),
         p99_kw=("power_kw", lambda s: np.percentile(s.dropna(), 99)), max_kw=("power_kw", "max"))
    .reset_index())
power_summary

In [ ]:
plot_signal = signals[(signals["crusher"] == selected_crusher) & (signals["signal_quality"] == "GOOD")].sort_values("timestamp").head(24*12*7)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_signal["timestamp"], plot_signal["power_kw"])
ax.set_title(f"Synthetic power signal example: {selected_crusher}")
ax.set_xlabel("Timestamp"); ax.set_ylabel("Power (kW)")
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(FIG_DIR / "06_selected_crusher_power_timeseries.png", dpi=150); plt.show()

## 8. Candidate modelling windows

Raw second-level observations should not be treated as independent. This demo builds 15-minute processing windows by aligning high-confidence truck dumps with crusher power.

In [ ]:
selected_trucks = trucks[(trucks["selected_crusher"] == selected_crusher) & (trucks["traceability_confidence"] == "HIGH") &
                         (trucks["origin_type"] == "POLYGON_DIRECT") & trucks["payload_tonnes"].notna() & trucks["dump_datetime"].notna()].copy()
selected_trucks["dump_datetime"] = pd.to_datetime(selected_trucks["dump_datetime"])
selected_trucks["window_15min"] = selected_trucks["dump_datetime"].dt.floor("15min")
truck_windows = selected_trucks.groupby("window_15min").agg(truck_cycles=("LOADID", "count"), linked_blasts=("blast_name", "nunique"), dumped_tonnes=("payload_tonnes", "sum")).reset_index().rename(columns={"window_15min": "timestamp"})

sig = signals[(signals["crusher"] == selected_crusher) & (signals["signal_quality"] == "GOOD") & (signals["operational_state"] == "RUNNING")].copy()
sig["timestamp"] = pd.to_datetime(sig["timestamp"])
sig["window_15min"] = sig["timestamp"].dt.floor("15min")
power_windows = sig.groupby("window_15min").agg(mean_power_kw=("power_kw", "mean"), p95_power_kw=("power_kw", lambda s: np.percentile(s.dropna(), 95)), p99_power_kw=("power_kw", lambda s: np.percentile(s.dropna(), 99)), max_power_kw=("power_kw", "max"), signal_records=("power_kw", "count")).reset_index().rename(columns={"window_15min": "timestamp"})

modelling_windows = truck_windows.merge(power_windows, on="timestamp", how="inner")
modelling_windows["energy_kwh"] = modelling_windows["mean_power_kw"] * 0.25
modelling_windows["specific_energy_kwh_t"] = modelling_windows["energy_kwh"] / modelling_windows["dumped_tonnes"]
modelling_windows["peak_event_flag"] = modelling_windows["max_power_kw"] > modelling_windows["p99_power_kw"].median()
modelling_windows.head()

In [ ]:
window_summary = pd.DataFrame({
    "metric": ["candidate_windows", "total_dumped_tonnes", "mean_truck_cycles_per_window", "mean_linked_blasts_per_window", "mean_power_kw", "p95_power_kw", "p99_power_kw", "mean_specific_energy_kwh_t", "peak_event_rate_pct"],
    "value": [len(modelling_windows), modelling_windows["dumped_tonnes"].sum(), modelling_windows["truck_cycles"].mean(), modelling_windows["linked_blasts"].mean(), modelling_windows["mean_power_kw"].mean(), modelling_windows["p95_power_kw"].mean(), modelling_windows["p99_power_kw"].mean(), modelling_windows["specific_energy_kwh_t"].mean(), modelling_windows["peak_event_flag"].mean()*100]
})
window_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(modelling_windows["specific_energy_kwh_t"].dropna(), bins=50)
ax.set_title("Synthetic specific energy distribution")
ax.set_xlabel("Specific energy (kWh/t)"); ax.set_ylabel("Candidate windows")
fig.tight_layout(); fig.savefig(FIG_DIR / "07_specific_energy_distribution.png", dpi=150); plt.show()

## 9. Methodological decision table

Online learning is not assumed. It remains a design hypothesis that must beat scheduled retraining under chronological evaluation.

In [ ]:
method_decision_table = pd.DataFrame([
    {"strategy": "Static model", "role": "Minimum baseline", "evidence_needed": "Temporal holdout performance and error stability"},
    {"strategy": "Scheduled batch retraining", "role": "Industrial baseline", "evidence_needed": "Performance after monthly or quarterly updates"},
    {"strategy": "Online residual correction", "role": "Adaptive proposal", "evidence_needed": "Lower residual error or faster post-drift recovery"},
    {"strategy": "Drift-aware online adaptation", "role": "Full adaptive artifact", "evidence_needed": "Detected drift events and measurable improvement over scheduled retraining"},
])
method_decision_table

## 10. Dataset Datasheet and Model Card inputs

This EDA feeds two expected course artifacts:

**Dataset Datasheet:** motivation, composition, collection process, preprocessing, intended uses, prohibited uses, distribution, and maintenance.

**Model Card:** intended use, factors, metrics, evaluation data, training data, quantitative analyses, ethical considerations, caveats, and recommendations.

No predictive model is trained in this notebook; the analysis prepares the modelling cohort and reporting structure.

## 11. Generate Markdown EDA report

In [ ]:
report = f"""# Synthetic EDA Summary

## Scientific integrity note

This report was generated from synthetic data calibrated from verified aggregate operational statistics. It does not contain raw operational records.

## Verified aggregate inventory

- Study period: {STUDY_START} to {STUDY_END}
- Unique fired blasts: {sum(c['blasts'] for c in AGGREGATES.values()):,}
- Blast-hole records: {sum(c['holes'] for c in AGGREGATES.values()):,}
- Weighted mean holes per blast: {sum(c['holes'] for c in AGGREGATES.values()) / sum(c['blasts'] for c in AGGREGATES.values()):.2f}

## Synthetic tables generated

- synthetic_blasts.csv: {blasts.shape[0]:,} rows
- synthetic_truck_cycles.csv: {trucks.shape[0]:,} rows
- synthetic_crusher_signals.csv: {signals.shape[0]:,} rows

## Recommended primary crusher for initial confirmatory analysis

- {selected_crusher}

## Candidate modelling windows

- 15-minute high-confidence windows: {len(modelling_windows):,}
- Synthetic dumped tonnes in candidate windows: {modelling_windows['dumped_tonnes'].sum():,.2f}
- Mean specific energy: {modelling_windows['specific_energy_kwh_t'].mean():.6f} kWh/t

## Methodological conclusion

The EDA supports a staged modelling strategy: static baseline, scheduled batch retraining, online residual correction, and drift-aware adaptation. Online learning should only be retained if it outperforms scheduled retraining under chronological evaluation.
"""
(REPORT_DIR / "synthetic_eda_summary.md").write_text(report, encoding="utf-8")
print(report)

## 12. Reproducibility checklist

- Fixed seed.
- Verified aggregate inputs documented.
- Synthetic data clearly separated from real evidence.
- CSV outputs generated.
- Figures generated.
- Markdown report generated.
- No confidential raw data included.
- No real mine name, coordinates, source-system name, or equipment identifier exposed.